# Mastering XGBoost
Welcome to the guide on XGBoost.
### Kaggle Dataset:
[Titanic Dataset](https://www.kaggle.com/c/titanic)


### Why and What: Install Package
**WHY:** Required library.
**WHAT:** pip install.


In [ ]:
!pip install xgboost -q


### Why and What: Imports
**WHY:** Libraries for data manipulation.
**WHAT:** pandas, numpy, sklearn, matplotlib.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')


## Part 1: Theory Recap

XGBoost is famous for dominating tabular data competitions. Here is why.

### Real-World Analogy
You have a team of highly trained specialists. Each specialist reviews the errors of the team, calculates exactly how much to adjust to fix the error (using 2nd order derivatives), but they are strictly penalized if they try to memorize the answers (regularization). 

### Core Mathematics

1. **Objective Function:**
$$ \text{Obj} = \sum_{i=1}^n L(y_i, \hat{y}_i) + \sum_{k=1}^K \Omega(f_k) $$
* $L$ : Differentiable convex loss function measuring difference between target $y_i$ and prediction $\hat{y}_i$.
* $\Omega(f_k)$ : Regularization term penalizing the complexity of tree $k$.

2. **Optimal Leaf Weight:**
$$ w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda} $$
* $w_j^*$ : The optimal prediction weight for leaf $j$.
* $g_i$ : First-order gradient (derivative) of the loss function.
* $h_i$ : Second-order gradient (Hessian) of the loss function.
* $\lambda$ : L2 regularization term. If $\lambda$ is large, the leaf weight shrinks towards 0.

3. **Split Gain (To evaluate tree splits):**
$$ \text{Gain} = \frac{1}{2} \left[ \frac{(\sum_{L} g_i)^2}{\sum_{L} h_i + \lambda} + \frac{(\sum_{R} g_i)^2}{\sum_{R} h_i + \lambda} - \frac{(\sum_{I} g_i)^2}{\sum_{I} h_i + \lambda} \right] - \gamma $$
* $\gamma$ : Minimum loss reduction required to make a split (L1 regularization on leaves).


### Why and What: Loading Data
**WHY:** Real world data.
**WHAT:** Titanic.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
display(df.head())


### Why and What: Preprocessing
**WHY:** ML models need clean numerical data.
**WHAT:** Fillna, encode, scale.


In [ ]:
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


## Part 2: From Scratch Implementation
### Why and What: Scratch
**WHY:** Demystify the black box.
**WHAT:** Numpy implementation.


In [ ]:
class ScratchModel:
    def __init__(self, n=10, lr=0.1, reg=1.0):
        self.n=n; self.lr=lr; self.reg=reg; self.trees=[]
    def fit(self, X, y):
        pred = np.zeros(y.shape[0])
        for _ in range(self.n):
            p = 1 / (1 + np.exp(-pred))
            g, h = p - y, p * (1 - p)
            best_gain = 0; best_feat = 0; best_thresh = 0; left_val = 0; right_val = 0
            G, H = np.sum(g), np.sum(h)
            for feat in range(X.shape[1]):
                for thresh in np.unique(X[:, feat]):
                    mask = X[:, feat] < thresh
                    G_L, H_L = np.sum(g[mask]), np.sum(h[mask])
                    gain = 0.5 * ((G_L**2/(H_L+self.reg)) + ((G-G_L)**2/(H-H_L+self.reg)) - (G**2/(H+self.reg)))
                    if gain > best_gain:
                        best_gain, best_feat, best_thresh = gain, feat, thresh
                        left_val, right_val = -G_L/(H_L+self.reg), -(G-G_L)/(H-H_L+self.reg)
            self.trees.append((best_feat, best_thresh, left_val, right_val))
            mask = X[:, best_feat] < best_thresh
            pred[mask] += self.lr * left_val; pred[~mask] += self.lr * right_val
    def predict_proba(self, X):
        pred = np.zeros(X.shape[0])
        for feat, thresh, left_val, right_val in self.trees:
            mask = X[:, feat] < thresh
            pred[mask] += self.lr * left_val; pred[~mask] += self.lr * right_val
        return 1 / (1 + np.exp(-pred))
    def predict(self, X): return (self.predict_proba(X) >= 0.5).astype(int)


### Why and What: Evaluation
**WHY:** Verify correctness.
**WHAT:** Predict and accuracy.


In [ ]:
scratch_model = ScratchModel()
scratch_model.fit(X_train, y_train)
y_pred_custom = scratch_model.predict(X_test)
print(f"Scratch Accuracy: {accuracy_score(y_test, y_pred_custom)*100:.2f}%")


## Part 3: Library Implementation
### Why and What: Library
**WHY:** Optimized production code.
**WHAT:** Official package.


In [ ]:
from xgboost import XGBClassifier
model_library = XGBClassifier(n_estimators=50, eval_metric='logloss', random_state=42)
model_library.fit(X_train, y_train)
print('XGBoost Accuracy:', accuracy_score(y_test, model_library.predict(X_test)))


### Why and What: Visualizations
**WHY:** Diagnose behavior.
**WHAT:** ROC and Importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if hasattr(model_library, 'predict_proba'):
    fpr, tpr, _ = roc_curve(y_test, model_library.predict_proba(X_test)[:, 1])
    axes[0].plot(fpr, tpr, color='darkorange', label=f'ROC (area = {auc(fpr, tpr):.2f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(loc='lower right')
importances = getattr(model_library, 'feature_importances_', None)
if importances is not None:
    idx = np.argsort(importances)
    axes[1].barh(range(len(idx)), importances[idx], color='teal', label='Importance')
    axes[1].set_yticks(range(len(idx)))
    axes[1].set_yticklabels(df_clean.drop('Survived', axis=1).columns[idx])
    axes[1].legend(loc='lower right')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance'); axes[1].set_ylabel('Features')
plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments
### Why and What: Tuning
**WHY:** Optimize model.
**WHAT:** Grid search.


In [ ]:
res=[]
for lr in [0.01, 0.1]:
 for n in [10, 50, 100]:
  clf=XGBClassifier(n_estimators=n, learning_rate=lr, eval_metric='logloss')
  res.append({'n':n, 'lr':lr, 'acc':cross_val_score(clf, X_scaled, y, cv=3).mean()})
sns.lineplot(data=pd.DataFrame(res), x='n', y='acc', hue='lr')
plt.title('XGBoost Hyperparameters'); plt.legend(title='LR'); plt.show()


## Part 5: Interview Corner
**Q: How does XGBoost handle missing values?**
**A:** Sparsity-aware split finding. It assigns missing values to the child node that minimizes the loss.


## Key Takeaways
- Taylor Expansion.
- Regularization.
- Scalability.
- Sparsity Aware.
- Fast execution.
